# LIDAR vs MintPy

This workflow masks and subsets the MintPy timeseries, subsets `geometryGeo.h5`, projects the LIDAR field into InSAR LOS, and then computes correlations against MintPy displacement slices.

## Overview

Compares LIDAR snow depth measurements against MintPy InSAR displacement timeseries. Projects LIDAR measurements into InSAR line-of-sight (LOS) and computes spatial correlations.

**Prerequisites:**
- MintPy installed with geocoded products
- Resampled LIDAR GeoTIFF (output from `resample_LIDAR_to_MintPy.ipynb`)
- `snowsar` package installed

**Workflow:**
1. Configure paths and analysis parameters
2. Mask and subset InSAR timeseries to LIDAR coverage
3. Project LIDAR field into InSAR LOS geometry
4. Compute correlation statistics

**Recommended directory structure:**
```
project/
├── data/
│   ├── mintpy/
│   │   ├── geo_timeseries_demErr.h5     # InSAR displacement
│   │   └── geo_geometryRadar.h5          # LOS geometry
│   └── lidar_resampled/
│       └── resampled_*.tif               # From previous notebook
└── outputs/
    ├── *_masked.h5                       # Created automatically
    ├── *_subset.h5
    └── *_subset.tif
```

**Interpreting correlation results:**
- **r > 0.7**: Strong positive correlation (LIDAR and InSAR agree well)
- **0.4 < r < 0.7**: Moderate correlation (general agreement with noise)
- **r < 0.4**: Weak correlation (investigate data quality, projection, or temporal mismatch)
- **p-value < 0.05**: Statistically significant result

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from mintpy.cli import generate_mask, mask as mintpy_mask, subset, view
from mintpy.utils import utils as ut

from snowsar.utils.lidar_utils import (
    compute_pearson_correlation,
    cumulative_sum_through_date,
    list_hdf5_root_datasets,
    local_incidence_from_geometry,
    mintpy_date_from_slice_name,
    mintpy_date_slice_names,
    project_scalar_field_to_los,
    resample_geometry_dataset_to_raster,
    subset_geotiff_by_bbox,
)


In [ ]:
# ============================================================================
# CONFIGURATION - Update these paths for your setup
# ============================================================================

NOTEBOOK_DIR = Path.cwd() / "LIDAR_notebooks" if (Path.cwd() / "LIDAR_notebooks").exists() else Path.cwd()
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Path to MintPy geocoded timeseries (InSAR displacement in LOS)
TIMESERIES_FILE = Path("./data/mintpy/geo_timeseries_demErr.h5")

# Path to MintPy geometry file (contains incidence/azimuth angles for LOS projection)
GEOMETRY_FILE = Path("./data/mintpy/geo_geometryRadar.h5")

# Path to resampled LIDAR file (output from resample_LIDAR_to_MintPy notebook)
LIDAR_FILE = Path("./data/lidar_resampled/resampled_ASO_Truckee_2023Apr09_snowdepth_3m-006.tif")

# InSAR acquisition date to compare (format: YYYYMMDD)
TARGET_DATE = "20230406"

# End date for cumulative displacement analysis (format: YYYYMMDD)
CUMULATIVE_END_DATE = "20230406"

# ============================================================================
# ANALYSIS PARAMETERS
# ============================================================================

# LIDAR product type (determines what's being measured)
# Options: "snow_depth", "snow_water_equivalent"
LIDAR_PRODUCT_NAME = "snow_depth"

# Direction of LIDAR measurement
# - "vertical": straight up/down (typical for airborne LIDAR)
# - "surface_normal": perpendicular to local terrain (for steep slopes)
# Use "vertical" unless terrain slope exceeds ~30° in your study area
LIDAR_DIRECTION = "vertical"

# Mask radar shadow regions (areas blocked from satellite view by terrain)
# Set True if your area has steep topography; False for gentle terrain
MASK_SHADOW = False

# ============================================================================
# SUBSET REGION (speeds up processing, reduces memory)
# ============================================================================
# Latitude range (min, max) in decimal degrees
SUBSET_LAT = (38.68, 39.65)

# Longitude range (min, max) in decimal degrees
SUBSET_LON = (-120.50, -119.78)

# ============================================================================
# DISPLAY OPTIONS
# ============================================================================
# Scale factor to convert displacement to cm (1 = already in cm)
DISPLACEMENT_SCALE_CM = 1

# ============================================================================
# Validation
# ============================================================================
for path in (TIMESERIES_FILE, GEOMETRY_FILE, LIDAR_FILE):
    if not path.exists():
        raise FileNotFoundError(
            f"Input file not found: {path}\n"
            f"Update the configuration paths above."
        )

# Intermediate output paths (created automatically)
LIDAR_MASK_FILE = OUTPUT_DIR / f"{LIDAR_FILE.stem}_mask.h5"
MASKED_TIMESERIES_FILE = OUTPUT_DIR / f"{TIMESERIES_FILE.stem}_masked.h5"
SUBSET_TIMESERIES_FILE = OUTPUT_DIR / f"{TIMESERIES_FILE.stem}_masked_subset.h5"
SUBSET_GEOMETRY_FILE = OUTPUT_DIR / f"{GEOMETRY_FILE.stem}_subset.h5"
SUBSET_LIDAR_FILE = OUTPUT_DIR / f"{LIDAR_FILE.stem}_subset{LIDAR_FILE.suffix}"

print("Configuration validated. Ready to process.")

## Generate mask and subset products

In [ ]:
generate_mask.main([str(LIDAR_FILE), "-o", str(LIDAR_MASK_FILE)])
mintpy_mask.main([
    str(TIMESERIES_FILE),
    "--mask", str(LIDAR_MASK_FILE),
    "--outfile", str(MASKED_TIMESERIES_FILE),
])
subset.main([
    str(MASKED_TIMESERIES_FILE),
    "--lat", str(SUBSET_LAT[0]), str(SUBSET_LAT[1]),
    "--lon", str(SUBSET_LON[0]), str(SUBSET_LON[1]),
    "-o", str(SUBSET_TIMESERIES_FILE),
])
subset.main([
    str(GEOMETRY_FILE),
    "--lat", str(SUBSET_LAT[0]), str(SUBSET_LAT[1]),
    "--lon", str(SUBSET_LON[0]), str(SUBSET_LON[1]),
    "-o", str(SUBSET_GEOMETRY_FILE),
])
subset_geotiff_by_bbox(
    LIDAR_FILE,
    SUBSET_LIDAR_FILE,
    lat_range=SUBSET_LAT,
    lon_range=SUBSET_LON,
)


## Inspect geometry and LOS projection

In [ ]:
available_geometry = list_hdf5_root_datasets(SUBSET_GEOMETRY_FILE)
pd.DataFrame({"dataset": available_geometry})


In [ ]:
with rasterio.open(SUBSET_LIDAR_FILE) as src:
    lidar_data = src.read(1).astype(np.float32)
    if src.nodata is not None:
        lidar_data = np.where(lidar_data == src.nodata, np.nan, lidar_data)

lidar_los = project_scalar_field_to_los(
    lidar_data,
    SUBSET_GEOMETRY_FILE,
    quantity_direction=LIDAR_DIRECTION,
    mask_shadow=MASK_SHADOW,
    target_raster_path=SUBSET_LIDAR_FILE,
)
projection_summary = {
    "lidar_product": LIDAR_PRODUCT_NAME,
    "quantity_direction": lidar_los["quantity_direction"],
    "projection_mode": lidar_los["projection_mode"],
    "reason": lidar_los["reason"],
    "mask_shadow": MASK_SHADOW,
}
pd.DataFrame([projection_summary])


In [ ]:
if lidar_los["projection_mode"] == "terrain":
    local_inc = local_incidence_from_geometry(SUBSET_GEOMETRY_FILE)
    pd.DataFrame([
        {
            "local_incidence_mean_deg": float(np.nanmean(local_inc["local_incidence_deg"])),
            "local_incidence_std_deg": float(np.nanstd(local_inc["local_incidence_deg"])),
            "projection_factor_mean": float(np.nanmean(lidar_los["projection_factor"])),
        }
    ])
else:
    pd.DataFrame([
        {
            "projection_factor_mean": float(np.nanmean(lidar_los["projection_factor"])),
            "note": lidar_los["reason"] or "Flat incidence projection used.",
        }
    ])


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
if lidar_los["projection_mode"] == "terrain":
    image = local_inc["local_incidence_deg"]
    title = "Local Incidence Angle"
    cmap = "rainbow"
    cbar_label = "Local incidence angle (deg)"
    vmin, vmax = 0, 90
else:
    incidence_on_lidar_grid = resample_geometry_dataset_to_raster(
        SUBSET_GEOMETRY_FILE,
        "incidenceAngle",
        SUBSET_LIDAR_FILE,
    )
    image = incidence_on_lidar_grid
    title = "Incidence Angle on LIDAR Grid (terrain mode unavailable)"
    cmap = "rainbow"
    cbar_label = "Incidence angle (deg)"
    vmin, vmax = None, None

im = plt.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
plt.colorbar(im, label=cbar_label)
plt.title(title)
plt.xlabel("Column")
plt.ylabel("Row")
plt.tight_layout()
plt.show()


## Quick visual checks

In [ ]:
view.main([
    str(SUBSET_TIMESERIES_FILE), TARGET_DATE,
    "--sub-lat", str(SUBSET_LAT[0]), str(SUBSET_LAT[1]),
    "--sub-lon", str(SUBSET_LON[0]), str(SUBSET_LON[1]),
    "-v", "-2", "10",
    "--noverbose",
    "-c", "rainbow",
    "--math", "reverse",
    # "--nodisplay"
])

view.main([
    str(SUBSET_LIDAR_FILE),
    "--sub-lat", str(SUBSET_LAT[0]), str(SUBSET_LAT[1]),
    "--sub-lon", str(SUBSET_LON[0]), str(SUBSET_LON[1]),
    "-v", "0", "5",
    # "-u", "m",
    "-c", "rainbow",
    "--noreference",
    "--noverbose",
    # "--nodisplay"
])


## Correlation analysis with LOS-projected LIDAR

In [ ]:
def calculate_swe(ts_arr, inc_angle):
    F = (-0.6784*inc_angle**2)+(0.2899*inc_angle)-0.8473
    return ts_arr/-F

In [ ]:
target_dataset = f"timeseries-{TARGET_DATE}"
ts_data, _ = ut.readfile.read(SUBSET_TIMESERIES_FILE, datasetName=target_dataset)

target_result = compute_pearson_correlation(
    lidar_los["projected_los"],
    ts_data * DISPLACEMENT_SCALE_CM,
    on_invalid="nan",
)
pd.DataFrame([{"dataset": target_dataset, **target_result}])


**Single date correlation:**
Compares LIDAR measurement against InSAR displacement on `TARGET_DATE`.

Strong correlation (r > 0.7) indicates LIDAR snow loading matches InSAR ground deformation.

In [ ]:
ts_data_swe = calculate_swe(-ts_data, incidence_on_lidar_grid)
target_result_swe = compute_pearson_correlation(
    lidar_los["projected_los"],
    ts_data_swe * DISPLACEMENT_SCALE_CM,
    on_invalid="nan",
)
pd.DataFrame([{"dataset": target_dataset, **target_result_swe}])

In [ ]:
slice_names = mintpy_date_slice_names(SUBSET_TIMESERIES_FILE)
slice_dates = [mintpy_date_from_slice_name(name) for name in slice_names]
ts_stack = [ut.readfile.read(SUBSET_TIMESERIES_FILE, datasetName=name)[0] for name in slice_names]
ts_stack = np.stack(ts_stack)

cumulative_ts = cumulative_sum_through_date(ts_stack, slice_dates, CUMULATIVE_END_DATE)
cumulative_result = compute_pearson_correlation(
    lidar_los["projected_los"],
    cumulative_ts * DISPLACEMENT_SCALE_CM,
    on_invalid="nan",
)
pd.DataFrame([{"dataset": f"through_{CUMULATIVE_END_DATE}", **cumulative_result}])


**Cumulative displacement correlation:**
Sums all InSAR displacement from first acquisition through `CUMULATIVE_END_DATE`.

Better for seasonal snow accumulation (integrates multiple acquisitions).

In [ ]:
rows = []
for slice_name in slice_names:
    ts_slice, _ = ut.readfile.read(SUBSET_TIMESERIES_FILE, datasetName=slice_name)
    result = compute_pearson_correlation(
        lidar_los["projected_los"],
        ts_slice * DISPLACEMENT_SCALE_CM,
        on_invalid="nan",
    )
    rows.append({"dataset": slice_name, "date": mintpy_date_from_slice_name(slice_name), **result})

pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


**Time series of correlations:**
Correlation for each InSAR acquisition date. Helps identify:
- Best/worst dates for comparison
- Temporal decorrelation patterns
- Optimal acquisition timing